In [8]:
import pandas as pd
import requests
import os

# Download using requests with SSL verification disabled
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

print("Downloading dataset...")
response = requests.get(url, verify=False)

# Save the raw excel file
os.makedirs("../data/raw", exist_ok=True)
with open("../data/raw/online_retail.xlsx", "wb") as f:
    f.write(response.content)

# Now read from local file
df = pd.read_excel("../data/raw/online_retail.xlsx")
df.to_csv("../data/raw/online_retail_raw.csv", index=False)

print(f"Downloaded! Shape: {df.shape}")
print(df.head())

/Users/mdkamrulhasan/Desktop/ecommerce-sales-analytics/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'archive.ics.uci.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Downloaded! Shape: (541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [9]:
print("=== Dataset Overview ===")
print(f"Total transactions: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Total countries: {df['Country'].nunique()}")
print(f"Total unique customers: {df['CustomerID'].nunique():,}")
print(f"Total unique products: {df['StockCode'].nunique():,}")

print(f"\n=== Missing Values ===")
print(df.isnull().sum())

print(f"\n=== Data Types ===")
print(df.dtypes)

=== Dataset Overview ===
Total transactions: 541,909
Total columns: 8

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Total countries: 38
Total unique customers: 4,372
Total unique products: 4,070

=== Missing Values ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

=== Data Types ===
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [14]:
import os
os.makedirs("../data/processed", exist_ok=True)

print(f"Shape before cleaning: {df.shape}")

# 1. Drop rows with missing CustomerID
df_clean = df.dropna(subset=['CustomerID'])

# 2. Drop rows with missing Description
df_clean = df_clean.dropna(subset=['Description'])

# 3. Remove cancelled orders (InvoiceNo starts with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# 4. Remove negative quantities (returns/errors)
df_clean = df_clean[df_clean['Quantity'] > 0]

# 5. Remove zero or negative unit prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# 6. Fix CustomerID type
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 7. Add revenue column
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 8. Add date parts for analysis
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

# Save cleaned data
df_clean.to_csv("../data/processed/online_retail_clean.csv", index=False)

print(f"Shape after cleaning: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]:,}")
print(f"\nTotal Revenue: ${df_clean['Revenue'].sum():,.2f}")
print(f"Average Order Value: ${df_clean.groupby('InvoiceNo')['Revenue'].sum().mean():,.2f}")
print("\n✅ Cleaned data saved!")

Shape before cleaning: (541909, 8)
Shape after cleaning: (397884, 13)
Rows removed: 144,025

Total Revenue: $8,911,407.90
Average Order Value: $480.87

✅ Cleaned data saved!


In [15]:
import sqlite3

# Create database
conn = sqlite3.connect("../data/processed/ecommerce.db")

# Load clean data into SQL table
df_clean.to_sql("transactions", conn, if_exists="replace", index=False)
print("✅ Data loaded into SQLite!")

# Query 1 — Total revenue by country (top 10)
q1 = pd.read_sql("""
    SELECT 
        Country,
        COUNT(DISTINCT InvoiceNo) as total_orders,
        COUNT(DISTINCT CustomerID) as total_customers,
        ROUND(SUM(Revenue), 2) as total_revenue,
        ROUND(AVG(Revenue), 2) as avg_order_value
    FROM transactions
    GROUP BY Country
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)

print("\n📊 Top 10 Countries by Revenue:")
print(q1.to_string(index=False))

✅ Data loaded into SQLite!

📊 Top 10 Countries by Revenue:
       Country  total_orders  total_customers  total_revenue  avg_order_value
United Kingdom         16646             3920     7308391.55            20.63
   Netherlands            94                9      285446.34           121.00
          EIRE           260                3      265545.90            36.70
       Germany           457               94      228867.14            25.32
        France           389               87      209024.05            25.06
     Australia            57                9      138521.31           117.19
         Spain            90               30       61577.11            24.79
   Switzerland            51               21       56443.95            30.66
       Belgium            98               25       41196.34            20.28
        Sweden            36                8       38378.33            85.10


In [16]:
# Query 2 — Monthly revenue trend
q2 = pd.read_sql("""
    SELECT 
        Year,
        Month,
        COUNT(DISTINCT InvoiceNo) as total_orders,
        ROUND(SUM(Revenue), 2) as monthly_revenue,
        COUNT(DISTINCT CustomerID) as unique_customers
    FROM transactions
    GROUP BY Year, Month
    ORDER BY Year, Month
""", conn)

print("📊 Monthly Revenue Trend:")
print(q2.to_string(index=False))

# Query 3 — Top 10 best selling products
q3 = pd.read_sql("""
    SELECT 
        StockCode,
        Description,
        SUM(Quantity) as total_units_sold,
        ROUND(SUM(Revenue), 2) as total_revenue,
        COUNT(DISTINCT CustomerID) as unique_buyers
    FROM transactions
    GROUP BY StockCode, Description
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)

print("\n📊 Top 10 Products by Revenue:")
print(q3.to_string(index=False))

conn.close()
print("\n✅ SQL analysis complete!")

📊 Monthly Revenue Trend:
 Year  Month  total_orders  monthly_revenue  unique_customers
 2010     12          1400        572713.89               885
 2011      1           987        569445.04               741
 2011      2           997        447137.35               758
 2011      3          1321        595500.76               974
 2011      4          1149        469200.36               856
 2011      5          1555        678594.56              1056
 2011      6          1393        661213.69               991
 2011      7          1331        600091.01               949
 2011      8          1280        645343.90               935
 2011      9          1755        952838.38              1266
 2011     10          1929       1039318.79              1364
 2011     11          2657       1161817.38              1664
 2011     12           778        518192.79               615

📊 Top 10 Products by Revenue:
StockCode                        Description  total_units_sold  total_revenu